In [1]:
import postprocessors as pp
import numpy as np
import h5py
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
data_dir = '/data/wallops/site/'
infile = data_dir + '20231031.2000.00.wal.0.antennas_iq.hdf5.site'
rawfile = data_dir + '20231031.2000.00.wal.0.rawacf.hdf5.site'
rawout = data_dir + 'test/test.rawacf.hdf5.site'
bfout = data_dir + 'test/test.bfiq.hdf5.site'
antout = data_dir + 'test/test.antennas_iq.hdf5.site'

# Test processing record-by-record using core functions

In [ ]:
def check_rec(rec, raw_rec):
    data = rec['data'][()].reshape(rec['data_dimensions'][()])
    beam_azms = rec['beam_azms'][()]
    freq_khz = rec.attrs['freq']
    antenna_spacing_m = 12.8016
    main_data = data[:16]
    intf_data = data[16:]
    beam_azms = np.rad2deg(np.arcsin(15.24 * np.sin(np.deg2rad(rec['beam_azms'][()])) / antenna_spacing_m))
    main_beams = []
    intf_beams = []
    for i in range(main_data.shape[1]):
        main_beams.append(pp.AntennasIQ2Bfiq.beamform(
            main_data[:, i], 
            beam_azms, 
            freq_khz, 
            main_data.shape[0], # number of antennas in array
            antenna_spacing_m,
            np.arange(main_data.shape[0])  # antenna indices corresponding to first dimension of main_data
        ))
        intf_beams.append(pp.AntennasIQ2Bfiq.beamform(
            intf_data[:, i], 
            beam_azms, 
            freq_khz, 
            intf_data.shape[0], # number of antennas in array
            antenna_spacing_m,
            np.arange(intf_data.shape[0])  # antenna indices corresponding to first dimension of intf_data
        ))
    main_bf_data = np.array(main_beams)
    intf_bf_data = np.array(intf_beams)

    record = dict()
    record['pulses'] = rec['pulses'][()]
    record['lags'] = pp.AntennasIQ2Bfiq.create_lag_table(record)
    record['pulse_phase_offset'] = rec['pulse_phase_offset'][()]
    record['tau_spacing'] = rec.attrs['tau_spacing']
    record['rx_sample_rate'] = rec.attrs['rx_sample_rate']
    record['num_samps'] = bf_data.shape[-1]
    record['first_range'] = 180.0
    record['first_range_rtt'] = pp.AntennasIQ2Bfiq.calculate_first_range_rtt(record)
    record['num_ranges'] = pp.AntennasIQ2Bfiq.get_number_of_ranges(record)

    main_acfs = pp.Bfiq2Rawacf.correlations_from_samples(
        main_bf_data,
        main_bf_data, 
        record
    )
    intf_acfs = pp.Bfiq2Rawacf.correlations_from_samples(
        intf_bf_data,
        intf_bf_data, 
        record
    )
    xcfs = pp.Bfiq2Rawacf.correlations_from_samples(
        intf_bf_data,
        main_bf_data, 
        record
    )
    raw_main_acfs = raw_rec['main_acfs'][()].reshape(raw_rec['correlation_dimensions'][()])
    raw_intf_acfs = raw_rec['intf_acfs'][()].reshape(raw_rec['correlation_dimensions'][()])
    raw_xcfs = raw_rec['xcfs'][()].reshape(raw_rec['correlation_dimensions'][()])
    avg_main_acfs = np.einsum('ijkl->jkl', main_acfs) / main_acfs.shape[0]
    avg_intf_acfs = np.einsum('ijkl->jkl', intf_acfs) / intf_acfs.shape[0]
    avg_xcfs = np.einsum('ijkl->jkl', xcfs) / xcfs.shape[0]
    if (
        np.allclose(raw_main_acfs, avg_main_acfs) and 
        np.allclose(raw_intf_acfs, avg_intf_acfs) and 
        np.allclose(raw_xcfs, avg_xcfs)
    ):
        print(True)
    else:
        print(max([
            (np.abs(raw_main_acfs - avg_main_acfs) / np.abs(avg_main_acfs)).max(),
            (np.abs(raw_intf_acfs - avg_intf_acfs) / np.abs(avg_intf_acfs)).max(),
            (np.abs(raw_xcfs - avg_xcfs) / np.abs(avg_xcfs)).max()
        ]))
    

In [ ]:
antgroup = h5py.File(infile, 'r')
rawgroup = h5py.File(rawfile, 'r')
recs = sorted(list(antgroup.keys()))
for rec in recs:
    print(f'{rec}  ', end='')
    check_rec(antgroup[rec], rawgroup[rec])

# Test processing class on entire file and verify

In [3]:
from postprocessors.sandbox.update_beam_dirs import UpdateBeamDirs

def check_rawacf_equal(rec1, rec2):
    main_acfs1 = rec1['main_acfs'][()].reshape(rec1['correlation_dimensions'][()])
    intf_acfs1 = rec1['intf_acfs'][()].reshape(rec1['correlation_dimensions'][()])
    xcfs1 = rec1['xcfs'][()].reshape(rec1['correlation_dimensions'][()])
    main_acfs2 = rec2['main_acfs'][()].reshape(rec2['correlation_dimensions'][()])
    intf_acfs2 = rec2['intf_acfs'][()].reshape(rec2['correlation_dimensions'][()])
    xcfs2 = rec2['xcfs'][()].reshape(rec2['correlation_dimensions'][()])
    if (
        np.allclose(main_acfs1, main_acfs2) and 
        np.allclose(intf_acfs1, intf_acfs2) and 
        np.allclose(xcfs1, xcfs2)
    ):
        print(True)
    else:
        print(max([
            (np.abs(main_acfs1 - main_acfs2) / np.abs(main_acfs2)).max(),
            (np.abs(intf_acfs1 - intf_acfs2) / np.abs(intf_acfs2)).max(),
            (np.abs(xcfs1 - xcfs2) / np.abs(xcfs2)).max()
        ]))

In [4]:
data_dir = '/data/wallops/site/'
infile = data_dir + '20231031.2000.00.wal.0.antennas_iq.hdf5.site'
rawfile = data_dir + '20231031.2000.00.wal.0.rawacf.hdf5.site'
outfile = data_dir + 'test.rawacf.hdf5.site'

processor = UpdateBeamDirs(infile, outfile, 'site', 'site')
processor.process_file(num_processes=1)

processed_group = h5py.File(outfile, 'r')
rawgroup = h5py.File(rawfile, 'r')
recs = sorted(list(processed_group.keys()))
for rec in recs:
    print(f'{rec}  ', end='')
    check_rawacf_equal(processed_group[rec], rawgroup[rec])


1698782400016  ==============================================] 100.00%True
1698782403687  True
1698782407230  True
1698782410647  True
1698782414190  True
1698782417738  True
1698782421155  True
1698782424696  True
1698782428240  True
1698782431655  1.4346218e-05
1698782435197  True
1698782438741  True
1698782442161  True
1698782445705  True
1698782449247  True
1698782452663  True
1698782460016  8.813349e-05
1698782463693  True
1698782467235  True
1698782470651  True
1698782474194  3.2029555e-05
1698782477738  1.2540757e-05
1698782481152  True
1698782484705  True
1698782488251  True
1698782491665  True
1698782495206  True
1698782498748  True
1698782502163  True
1698782505716  1.1690278e-05
1698782509264  True
1698782512680  2.132507e-05
1698782520017  True
1698782523687  True
1698782527229  True
1698782530652  True
1698782534194  True
1698782537735  True
1698782541153  2.793453e-05
1698782544694  4.362301e-05
1698782548235  True
1698782551661  1.6281025e-05
1698782555203  True
1698782